# Tutorial 8-1: Bagging vs. Single Decision Trees
**Course:** CSEN 140: Machine Learning/Data Mining  
**Instructor:** Dr. David C. Anastasiu

--- 

## 1. Introduction: The Power of the Ensemble
Decision trees are powerful but "unstable" learners. A small change in the training data can lead to a completely different tree structure, a phenomenon known as **high variance**. 

**Bagging (Bootstrap Aggregating)** addresses this by:
1. **Bootstrapping:** Creating multiple training sets by sampling from the original data with replacement.
2. **Aggregating:** Training a separate tree on each sample and combining their predictions via majority voting.

**Objective:** We will visualize the decision boundaries of a single overfit tree versus a Bagged ensemble to see how aggregating models "smooths out" the noise.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.datasets import make_moons

# Step 1: Create a noisy 'Moons' dataset
X, y = make_moons(n_samples=300, noise=0.3, random_state=42)

def plot_decision_boundary(clf, X, y, ax, title):
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02), np.arange(y_min, y_max, 0.02))
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    ax.scatter(X[:, 0], X[:, 1], c=y, edgecolors='k', cmap='RdBu')
    ax.set_title(title)

## 2. The Single "Unstable" Tree
We will fit a single, deep decision tree. Because we aren't pruning, the tree will try to capture every noisy outlier, leading to a jagged and complex decision boundary.

In [ ]:
tree_clf = DecisionTreeClassifier(random_state=42)
tree_clf.fit(X, y)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
plot_decision_boundary(tree_clf, X, y, ax[0], "Single Decision Tree (Unstable)")

# Placeholder for Step 3 calculation below
ax[1].set_axis_off()

## 3. The Bagging Ensemble
Now we build an ensemble of 500 identical trees. Each tree sees a slightly different version of the data. When they vote, the random noise "averages out," while the true underlying pattern (the crescent moons) remains.

In [ ]:
bag_clf = BaggingClassifier(
    DecisionTreeClassifier(random_state=42), n_estimators=500,
    max_samples=100, bootstrap=True, n_jobs=-1, random_state=42
)
bag_clf.fit(X, y)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
plot_decision_boundary(tree_clf, X, y, ax[0], "Single Decision Tree")
plot_decision_boundary(bag_clf, X, y, ax[1], "Bagging Ensemble (Aggregated)")
plt.show()

## Summary
* **Variance Reduction:** The Bagging boundary is much smoother and likely generalizes better to new data.
* **Parallelism:** Because each bootstrap sample is independent, Bagging can be trained in parallel across multiple CPU cores (`n_jobs=-1`).
* **Robustness:** A single outlier can change a single tree's boundary, but it would need to appear in a majority of the 500 bootstrap samples to affect the Bagged ensemble.